# T5 — Text-to-Text Transfer Transformer

A study refresher. T5 (Raffel et al., 2020, *"Exploring the Limits of Transfer Learning with a Unified Text-to-Text Transformer"*) reframes **every** NLP task — translation, classification, regression, summarization, QA — as **text in → text out**, run on a single encoder–decoder Transformer.

**Domain:** Architectures  ·  **runnable:** yes

## 1. What & Why

**What it is.** T5 is a vanilla **encoder–decoder Transformer** (the original 2017 architecture, with a few deliberate simplifications) pretrained on a giant cleaned web corpus (**C4**, the "Colossal Clean Crawled Corpus") with a **span-corruption denoising** objective. Its defining idea is the **text-to-text framework**: cast *every* problem as mapping an input string to an output string. There are no task-specific heads — a classification label, a regression score, a translation, and a summary are all just decoded token sequences.

**The problem it solves.** Before T5 the transfer-learning landscape was a zoo of incompatible recipes: BERT-style encoders with a classification head here, GPT-style decoders there, span-prediction heads, sequence-tagging heads. Each task needed bespoke plumbing. T5's contribution is twofold:
1. **A unified interface.** One model, one loss (teacher-forced cross-entropy over output tokens), one decoding procedure for all tasks. You change the *prompt prefix*, not the architecture.
2. **A rigorous empirical study.** The paper is really a giant ablation: it systematically compares architectures, objectives, corpora, and fine-tuning strategies on equal footing, and the "T5 recipe" is the configuration that won.

**When to reach for it.** Any **seq2seq** problem — summarization, translation, data-to-text, grammar correction, question answering with generated answers, or turning structured labels into text. It's also a strong, well-understood baseline when you want an encoder–decoder rather than a decoder-only LLM. Its descendants (**FLAN-T5** for instruction tuning, **mT5** for 100+ languages, **ByT5** for tokenizer-free byte input, **T5X/UL2** for scale) are usually what you actually deploy today.

**When NOT to.** For open-ended chat / long-form generation, modern decoder-only models (the GPT/Llama/Claude family) dominate. For pure embedding/classification where you never generate text, a BERT-style encoder is cheaper. And the original T5's max length (512 in, 512 out by default) makes long-document work awkward without a long-context variant (LongT5).

## 2. Mental Model

**T5 is "fill in the blanks," scaled up and made universal.**

Think of pretraining as a cloze test on the entire internet: take a sentence, **rip out random spans**, replace each ripped span with a single numbered placeholder (a *sentinel*: `<extra_id_0>`, `<extra_id_1>`, …), and ask the model to **regenerate only the missing pieces**, in order, each tagged with its sentinel. That single skill — "given mangled text, reconstruct what's missing" — turns out to transfer to almost everything.

Then for any downstream task you keep the exact same machine and just **phrase the task as text**:

```
translate English to German: That is good.   ->   Das ist gut.
cola sentence: The cat sitted on mat.        ->   not acceptable     (grammaticality = a word!)
stsb sentence1: ... sentence2: ...           ->   3.8                (similarity = a number-as-text!)
summarize: <article>                         ->   <abstract>
```

The encoder reads the whole input bidirectionally (like BERT); the decoder generates the answer left-to-right (like GPT), attending back to the encoder. **The prefix is the API.** You don't add a head for "regression" — you train the model to literally emit the string `"3.8"`.

One more structural quirk to hold in your head: T5 throws away absolute position embeddings entirely. Instead, attention gets a **learned scalar bias that depends on the relative distance between query and key**, bucketed (nearby distances get their own bucket, far ones share log-spaced buckets). This bias is shared across all layers — a small, elegant simplification that helps it generalize to lengths it didn't see in training.

## 3. Key Concepts

- **Text-to-text framework.** Every task = string → string. No task-specific output heads; the "label" is decoded text. This is the whole thesis.
- **Task prefix.** A short instruction prepended to the input (`"summarize: "`, `"translate English to German: "`) that tells the *same* weights which task to perform. (Note: these are fixed strings used during fine-tuning, *not* free-form natural-language instructions — that comes later with FLAN-T5.)
- **Span-corruption objective.** Pretraining denoiser: mask contiguous spans (default ~15% of tokens, mean span length 3), replace each span with one **sentinel token**, and have the decoder emit only the dropped spans. Much more compute-efficient than BERT's per-token masking because the *target sequence is short* (only the missing bits).
- **Sentinel tokens (`<extra_id_0>`…).** ~100 special tokens reserved as the placeholders for corrupted spans. They appear in both the corrupted input and the target.
- **Encoder–decoder (seq2seq).** Full bidirectional encoder + autoregressive decoder with cross-attention. Contrast with encoder-only (BERT) and decoder-only (GPT). T5's ablations found encoder–decoder + denoising best for transfer.
- **C4 (Colossal Clean Crawled Corpus).** ~750 GB of deduplicated, filtered Common Crawl text built for the paper. The data-cleaning recipe itself was a contribution.
- **Relative position bias.** No sinusoidal or learned absolute positions. Attention logits get `+bias[bucket(key_pos − query_pos)]`, a learned scalar per (head, bucket), **shared across layers**. Enables length extrapolation.
- **T5 simplifications vs. vanilla Transformer.** LayerNorm with **no bias and no mean-subtraction** (RMSNorm-style), applied *pre*-block; no bias terms in dense layers; relative position bias instead of absolute. These small changes matter for stability at scale.
- **Model sizes.** `t5-small` (60M), `base` (220M), `large` (770M), `3B`, `11B`. Plus variants: **FLAN-T5** (instruction-tuned, much better zero-shot), **mT5** (multilingual), **ByT5** (byte-level), **LongT5** (long inputs), **UL2** (mixture-of-denoisers).

## 4. Setup

The conceptual cells below need only **NumPy**, so they run anywhere on CPU. To run a real T5 model, install Hugging Face Transformers (and a backend such as PyTorch). `t5-small` is ~240 MB; that cell is gated so the notebook still executes top-to-bottom without the download or the dependency.

```bash
pip install numpy
pip install "transformers>=4.40" torch sentencepiece   # for the real-model cell
```

In [1]:
import numpy as np

# Only NumPy is required for the conceptual cells. We pin a seed for reproducibility.
rng = np.random.default_rng(0)
print("NumPy", np.__version__, "ready — conceptual T5 demos run on CPU.")

NumPy 2.4.3 ready — conceptual T5 demos run on CPU.


## 5. Worked Examples

Three runnable, dependency-light demos that capture T5's essence — (1) the text-to-text interface, (2) the span-corruption pretraining objective, (3) the relative-position attention bias — followed by an optional cell that drives the *real* `t5-small` model if Transformers is installed.

### Example 1 — The text-to-text interface

The entire point of T5: heterogeneous tasks become one signature, `str -> str`. Classification emits a *word*; regression emits a *number rendered as text*. Only the prefix changes.

In [2]:
# Each task is just a string-to-string mapping with a task prefix.
tasks = {
    "translate English to German: That is good.":            "Das ist gut.",
    "cola sentence: The course is jumping well.":            "not acceptable",
    "stsb sentence1: The cat sat. sentence2: A cat is sitting.": "3.8",
    "summarize: state authorities dispatched emergency crews ...": "state crews respond to flooding",
}
for inp, target in tasks.items():
    print(f"IN  : {inp}")
    print(f"OUT : {target}\n")

IN  : translate English to German: That is good.
OUT : Das ist gut.

IN  : cola sentence: The course is jumping well.
OUT : not acceptable

IN  : stsb sentence1: The cat sat. sentence2: A cat is sitting.
OUT : 3.8

IN  : summarize: state authorities dispatched emergency crews ...
OUT : state crews respond to flooding



### Example 2 — The span-corruption pretraining objective

Mask contiguous spans, replace each with one sentinel token, and have the decoder reconstruct **only** the dropped spans (each prefixed by its sentinel, with a trailing sentinel to close). Note how short the target is versus the input — that compute efficiency is a key reason this objective was chosen over BERT-style per-token masking.

In [3]:
def span_corrupt(text, mask_prob=0.15, seed=0):
    """Toy version of T5's denoising objective (word-level, not sub-word)."""
    rng = np.random.default_rng(seed)
    tokens = text.split()
    masked = rng.random(len(tokens)) < mask_prob
    enc_in, dec_target = [], []
    sentinel, prev = 0, False
    for tok, m in zip(tokens, masked):
        if m:
            if not prev:                       # start of a new span -> one sentinel
                tag = f"<extra_id_{sentinel}>"
                enc_in.append(tag)             # placeholder in the input
                dec_target.append(tag)         # same tag opens the target span
                sentinel += 1
            dec_target.append(tok)             # the dropped word goes only to the target
            prev = True
        else:
            enc_in.append(tok)
            prev = False
    dec_target.append(f"<extra_id_{sentinel}>")  # closing sentinel
    return " ".join(enc_in), " ".join(dec_target)

text = "Thank you for inviting me to your party last week"
enc_in, dec_target = span_corrupt(text)
print("original       :", text)
print("encoder input  :", enc_in)
print("decoder target :", dec_target)

original       : Thank you for inviting me to your party last week
encoder input  : Thank you <extra_id_0> me to your party last week
decoder target : <extra_id_0> for inviting <extra_id_1>


### Example 3 — Relative position bias

T5 has **no** absolute position embeddings. Instead each attention logit gets an additive, learned scalar that depends only on the *relative* distance `key_pos − query_pos`, mapped through log-spaced **buckets** (nearby offsets get exact buckets; far ones share). The bias table is `(num_heads, num_buckets)` and is **shared across all layers**. Below we reproduce the bucketing used by Hugging Face and show the resulting per-position bias for one head.

In [4]:
def relative_position_bucket(rel_pos, num_buckets=32, max_distance=128):
    """Bidirectional (encoder) bucketing, matching the T5 / HF implementation."""
    ret = np.zeros_like(rel_pos)
    n = -rel_pos
    num_buckets //= 2
    ret += (n < 0).astype(np.int64) * num_buckets   # sign goes to the upper half
    n = np.abs(n)
    max_exact = num_buckets // 2
    is_small = n < max_exact                         # small offsets: exact buckets
    large = max_exact + (
        np.log(n.astype(np.float64) / max_exact + 1e-9)
        / np.log(max_distance / max_exact) * (num_buckets - max_exact)
    ).astype(np.int64)                               # large offsets: log-spaced
    large = np.minimum(large, num_buckets - 1)
    ret += np.where(is_small, n, large)
    return ret

seq = 8
rel = np.arange(seq)[None, :] - np.arange(seq)[:, None]   # key_pos - query_pos, shape (q, k)
buckets = relative_position_bucket(rel)

bias_table = rng.normal(0, 0.1, size=(1, 32))             # (heads, buckets); one head shown
bias = bias_table[0, buckets]                            # (q, k) additive logit bias

print("bucket ids (indices into the shared learned bias table):")
print(buckets)
print("\nadditive attention bias for head 0 (rounded):")
print(np.round(bias, 3))

bucket ids (indices into the shared learned bias table):
[[ 0 17 18 19 20 21 22 23]
 [ 1  0 17 18 19 20 21 22]
 [ 2  1  0 17 18 19 20 21]
 [ 3  2  1  0 17 18 19 20]
 [ 4  3  2  1  0 17 18 19]
 [ 5  4  3  2  1  0 17 18]
 [ 6  5  4  3  2  1  0 17]
 [ 7  6  5  4  3  2  1  0]]

additive attention bias for head 0 (rounded):
[[ 0.013 -0.032  0.041  0.104 -0.013  0.137 -0.067  0.035]
 [-0.013  0.013 -0.032  0.041  0.104 -0.013  0.137 -0.067]
 [ 0.064 -0.013  0.013 -0.032  0.041  0.104 -0.013  0.137]
 [ 0.01   0.064 -0.013  0.013 -0.032  0.041  0.104 -0.013]
 [-0.054  0.01   0.064 -0.013  0.013 -0.032  0.041  0.104]
 [ 0.036 -0.054  0.01   0.064 -0.013  0.013 -0.032  0.041]
 [ 0.13   0.036 -0.054  0.01   0.064 -0.013  0.013 -0.032]
 [ 0.095  0.13   0.036 -0.054  0.01   0.064 -0.013  0.013]]


### Example 4 — Driving the real `t5-small` (optional / gated)

This loads the actual pretrained model from Hugging Face. It's gated behind both an import check and an `os.getenv("RUN_T5")` flag so the notebook still runs end-to-end without the ~240 MB download. Set `RUN_T5=1` and install `transformers torch sentencepiece` to execute it.

In [5]:
import os

if os.getenv("RUN_T5") == "1":
    from transformers import T5Tokenizer, T5ForConditionalGeneration

    tok = T5Tokenizer.from_pretrained("t5-small")
    model = T5ForConditionalGeneration.from_pretrained("t5-small")

    for prompt in [
        "translate English to German: The house is wonderful.",
        "summarize: " + (
            "the transformer architecture relies entirely on attention "
            "mechanisms, dispensing with recurrence and convolutions entirely."
        ),
    ]:
        ids = tok(prompt, return_tensors="pt").input_ids
        out = model.generate(ids, max_new_tokens=40)
        print(prompt[:45], "->", tok.decode(out[0], skip_special_tokens=True))
else:
    # Call shape shown without the download, so the notebook always executes.
    print("Skipped real model (set RUN_T5=1 and `pip install transformers torch "
          "sentencepiece`).")
    print("Expected with t5-small:")
    print("  translate English to German: The house is wonderful. -> Das Haus ist wunderbar.")

Skipped real model (set RUN_T5=1 and `pip install transformers torch sentencepiece`).
Expected with t5-small:
  translate English to German: The house is wonderful. -> Das Haus ist wunderbar.


## 6. Gotchas & Pitfalls

- **You must add the task prefix.** The public checkpoints were fine-tuned with specific prefixes (`"summarize: "`, `"translate English to German: "`, `"cola sentence: "`, …). Forget the prefix, or use a different one, and `t5-small/base/large` produce garbage. (FLAN-T5 is the variant that handles free-form natural-language instructions.)
- **`t5-small/base/large` are not chatbots.** They were trained on supervised task mixtures, not instruction-following or dialogue. For zero-shot instructions reach for **FLAN-T5**; for chat, a decoder-only model.
- **Regression targets are *text*.** STS-B similarity is decoded as the string `"3.8"`. You round to a small grid (0.2 increments) and parse it back. Occasionally the model emits an unparseable string — handle that.
- **Sentinel ordering matters.** In span corruption the target must list spans in input order, each opened by the matching `<extra_id_k>` and the sequence closed by the next sentinel. Mismatched sentinels = a broken objective.
- **`T5Tokenizer` needs SentencePiece.** `pip install sentencepiece`, or use `T5TokenizerFast`. A common first-run `ImportError`.
- **Length limits bite.** Default ~512 tokens in/out. Long documents silently truncate — use **LongT5** or chunk. Generation length is capped by `max_new_tokens`/`max_length`; summaries cut off if it's too low.
- **fp16 instability.** The original T5 (v1.0) was trained in bfloat16 and is numerically unstable in fp16 — activations overflow to NaN. Use **bf16** or fp32, or the **t5-v1.1 / FLAN** checkpoints which behave better.
- **t5-v1.1 ≠ t5-v1.0.** v1.1 uses GEGLU feed-forward, no embedding/softmax weight tying, and was pretrained on C4 *only* (no supervised mixture) — so v1.1 base **must be fine-tuned** before it's useful. Don't expect it to translate out of the box.
- **Decoder needs a start token.** Generation begins from the `pad` token as `decoder_start_token_id`. `model.generate()` handles this; hand-rolled loops often forget it.

## 7. When to Use vs Alternatives

| You want… | Reach for | Why / trade-off |
|---|---|---|
| **Summarization, translation, data→text, grammar fix** | **T5 / FLAN-T5** | Encoder–decoder is the natural fit for seq2seq; bidirectional encoder reads the full source, decoder generates conditioned on it. |
| **Zero-/few-shot from natural instructions** | **FLAN-T5** | Same architecture, instruction-tuned on 1.8k tasks — far better zero-shot than vanilla T5 at equal size. |
| **Multilingual (100+ languages)** | **mT5 / ByT5** | mT5 pretrained on mC4; ByT5 is tokenizer-free (bytes), robust to noise and rare scripts. |
| **Pure classification / NER / embeddings** | **BERT / RoBERTa / DeBERTa** | Encoder-only is cheaper — no autoregressive decoding. Generating a label string is overkill if you never need free text. |
| **Open-ended chat, long-form generation, in-context learning** | **Decoder-only LLM** (GPT/Llama/Claude) | Decoder-only scales better for generation and few-shot prompting; that's where the field went after ~2022. |
| **Very long inputs** | **LongT5 / UL2 / Pegasus-X** | Original T5's 512-token window and dense attention don't scale to long documents. |
| **One model, many supervised tasks, full fine-tune** | **T5** | The text-to-text interface makes multitask fine-tuning uniform; a great, reproducible research baseline. |

**Bottom line:** T5 is the canonical, well-documented **encoder–decoder** for supervised seq2seq and a superb baseline/teaching model. For instruction-following use FLAN-T5; for chat and open generation use a modern decoder-only LLM; for classification-only use a BERT-family encoder.

## 8. Resources

- **Paper — "Exploring the Limits of Transfer Learning with a Unified Text-to-Text Transformer"** (Raffel et al., 2020): https://arxiv.org/abs/1910.10683
- **Hugging Face T5 docs** (model classes, prefixes, variants, training tips): https://huggingface.co/docs/transformers/model_doc/t5
- **Google T5 repo (`google-research/text-to-text-transfer-transformer`)** — original code, C4, checkpoints: https://github.com/google-research/text-to-text-transfer-transformer
- **FLAN-T5 paper — "Scaling Instruction-Finetuned Language Models"** (Chung et al., 2022): https://arxiv.org/abs/2210.11416
- **mT5 paper** (multilingual T5): https://arxiv.org/abs/2010.11934
- **T5X / scaling recipes** (JAX reimplementation used for the large models): https://github.com/google-research/t5x

**Cross-links in this library:** [`transformer`](transformer.ipynb) (the base architecture), [`encoder-decoder`](encoder-decoder.ipynb) (the seq2seq pattern), [`bert`](bert.ipynb) (encoder-only contrast), [`attention-mechanisms`](attention-mechanisms.ipynb) (relative position bias).